# Classification with a single-layer perceptron

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from data_generation import generate_linearly_separable_data, plot_data, generate_non_linearly_separable_data

X, T = generate_linearly_separable_data()
plot_data(X, T, title="Linearly Separable Data")


### 1st algorithm: perceptron learning rule, sequential (online) mode 

In [ ]:
from algorithms import PerceptronRulePerceptron

def step_function(x):
    return np.where(x >= 0, 1, 0)

perceptron = PerceptronRulePerceptron(
                        input_size=2, 
                        output_size=1, 
                        learning_rate=0.09,
                        activation_function=step_function)

perceptron.train(X, T, epochs=100)

perceptron.plot_metrics()



### 2nd algorithm: delta (Widrow-Hoff) rule, sequential (online) mode 

In [ ]:
from algorithms import DeltaRulePerceptron

def delta_rule_activation_function(x):
    return np.where(x >= 0, 1, -1)

T_delta = np.where(T == 0, -1, T)

perceptron = DeltaRulePerceptron(
                        input_size=2, 
                        output_size=1, 
                        learning_rate=0.1,
                        activation_function=delta_rule_activation_function)

perceptron.train(X, T_delta, epochs=100)
print("Final weights:", perceptron.weights)
perceptron.plot_metrics()



### 3rd algorithm: delta (Widrow-Hoff) rule, batch mode, using the matrix formulation

In [ ]:
from algorithms import DeltaRuleBatchPerceptron

perceptron = DeltaRuleBatchPerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.1, 
                        activation_function=delta_rule_activation_function)

perceptron.train(X, T_delta, epochs=100)
print("Final weights:", perceptron.weights)
perceptron.plot_metrics()

In [ ]:
# test plotting function

def plot_metrics_comparison(perceptron_rule = None, delta_rule = None, delta_rule_batch = None, saved_fig_name = None):

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    if perceptron_rule is not None:
        plt.plot(perceptron_rule.mean_squared_error, label="Perceptron Rule")
    if delta_rule is not None:
        plt.plot(delta_rule.mean_squared_error, label="Delta Rule")
    if delta_rule_batch is not None:
        plt.plot(delta_rule_batch.mean_squared_error, label="Delta Rule Batch")
    plt.title("Mean Squared Error Comparison")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    if perceptron_rule is not None:
        plt.plot(perceptron_rule.misclassified_samples_rate, label="Perceptron Rule")
    if delta_rule is not None:
        plt.plot(delta_rule.misclassified_samples_rate, label="Delta Rule")
    if delta_rule_batch is not None:
        plt.plot(delta_rule_batch.misclassified_samples_rate, label="Delta Rule Batch")
    plt.title("Misclassified Samples Rate Comparison")
    plt.xlabel("Epochs")
    plt.ylabel("Misclassified Samples Rate")
    plt.legend()

    if saved_fig_name is not None:
        plt.savefig(saved_fig_name)

    plt.show()

### Test #1 Comparative Analysis - Delta learning rule, sequential vs Delta learning rule, batch mode

In [ ]:
X, T = generate_linearly_separable_data()

T_delta = np.where(T == 0, -1, T)

perceptron = DeltaRulePerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.01,
                        activation_function=delta_rule_activation_function)

perceptron_batch = DeltaRuleBatchPerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.05,
                        activation_function=delta_rule_activation_function)

plot_data(X,T_delta, title="Linearly Separable Data for Delta Rule")

perceptron.train(X, T_delta, epochs=200)
perceptron_batch.train(X, T_delta, epochs=200)

plot_metrics_comparison(perceptron_rule=None, delta_rule=perceptron, delta_rule_batch=perceptron_batch, saved_fig_name="metrics_comparison_test1.png")

# print the first time the perceptron rule and delta rule reach a misclassified samples rate of 0.0
print("First time perceptron rule reaches 0.0 misclassified samples rate:")
print(perceptron.get_first_zero_misclassification_epoch())

print("First time delta rule batch reaches 0.0 misclassified samples rate:")
print(perceptron_batch.get_first_zero_misclassification_epoch())

### Test #2 Can delta learning rule in batch mode converge without the bias? 

In [ ]:
from algorithms import DeltaRuleBatchPerceptron

X_no_bias,T_no_bias = generate_linearly_separable_data(mA = np.array([-.1, -.1]), mB = np.array([1, 1]), has_bias=False)


T_no = np.where(T_no_bias == 0, -1, T_no_bias)

perceptron_no_bias = DeltaRuleBatchPerceptron(
    input_size=2,
    output_size=1,
    learning_rate=0.001,
    activation_function=delta_rule_activation_function,
    has_bias=False
)
perceptron_no_bias.train(X_no_bias, T_no, epochs = 200)
print("Final weights=", perceptron_no_bias.weights)
perceptron_no_bias.plot_metrics()

### Test #3 Comparative Analysis - Classical Perceptron Learning Rule vs Delta Rule batch mode on a non-linearly separable dataset 

In [ ]:
X, T = generate_non_linearly_separable_data()

T_delta = np.where(T == 0, -1, T)

perceptron = PerceptronRulePerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.01,
                        activation_function=step_function)

perceptron_delta = DeltaRulePerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.01,
                        activation_function=delta_rule_activation_function)

plot_data(X,T_delta, title="Non-Linearly Separable Data for Delta Rule")

perceptron.train(X, T, epochs=100)
perceptron_delta.train(X, T_delta, epochs=100)

plot_metrics_comparison(perceptron_rule=perceptron, delta_rule=perceptron_delta, saved_fig_name="metrics_comparison_test3.png")

### Test #4 Perceptron learning rule, Delta learning rule both sequential and batch mode on the subsampled dataset

In [ ]:
def plot_metrics_comparison_multiple_run(first_run = None, second_run = None, third_run = None, plot_classes_accuracy=False, saved_fig_name = None):

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    if first_run is not None:
        plt.plot(first_run.mean_squared_error, label="First Run")
    if second_run is not None:
        plt.plot(second_run.mean_squared_error, label="Second Run")
    if third_run is not None:
        plt.plot(third_run.mean_squared_error, label="Third Run")
    plt.title("Mean Squared Error Comparison")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    if first_run is not None:
        plt.plot(first_run.misclassified_samples_rate, label="First Run")
    if second_run is not None:
        plt.plot(second_run.misclassified_samples_rate, label="Second Run")
    if third_run is not None:
        plt.plot(third_run.misclassified_samples_rate, label="Third Run")
    plt.title("Misclassified Samples Rate Comparison")
    plt.xlabel("Epochs")
    plt.ylabel("Misclassified Samples Rate")
    plt.legend()
    
    plt.show()
    
    if first_run is not None:
        plt.plot(first_run.misclassified_samples_A_rate, label="First Run - Class A")
        plt.plot(first_run.misclassified_samples_B_rate, label="First Run - Class B")
    if second_run is not None:
        plt.plot(second_run.misclassified_samples_A_rate, label="Second Run - Class A")
        plt.plot(second_run.misclassified_samples_B_rate, label="Second Run - Class B")
    if third_run is not None:
        plt.plot(third_run.misclassified_samples_A_rate, label="Third Run - Class A")
        plt.plot(third_run.misclassified_samples_B_rate, label="Third Run - Class B")
    plt.title("Misclassified Samples Rate by Class")
    plt.xlabel("Epochs")
    plt.ylabel("Misclassified Samples Rate")
    plt.legend()
    plt.show()
    if saved_fig_name is not None:
        plt.savefig(saved_fig_name)

    plt.show()

In [ ]:
# Test on second non-linearly separable dataset

from data_generation import generate_non_linearly_separable_data_second_version, plot_data
from algorithms import DeltaRuleBatchPerceptron

first_run = DeltaRuleBatchPerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.03,
                        activation_function=delta_rule_activation_function)

second_run = DeltaRuleBatchPerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.03,
                        activation_function=delta_rule_activation_function)

third_run = DeltaRuleBatchPerceptron(
                        input_size=2,
                        output_size=1,
                        learning_rate=0.03,
                        activation_function=delta_rule_activation_function)

X_og, T_og = generate_non_linearly_separable_data_second_version()


# remove 25% of the data points from each class

idx_A = np.where(T_og == 1)[0]
idx_B = np.where(T_og == 0)[0]

# Togliamo il 25% di ogni classe
remove_A = np.random.choice(idx_A, size=int(0.25 * len(idx_A)), replace=False)
remove_B = np.random.choice(idx_B, size=int(0.25 * len(idx_B)), replace=False)

remove_idx = np.concatenate([remove_A, remove_B])

# Tutti gli altri rimangono
keep_idx = np.setdiff1d(np.arange(len(T_og)), remove_idx)

X_sub = X_og[:, keep_idx]
T_sub = T_og[keep_idx]

T_delta = np.where(T_sub == 0, -1, T_sub)


plot_data(X_sub, T_sub, title="Second Non-Linearly Separable Dataset")
first_run.train(X_sub, T_delta, epochs=100)
second_run.train(X_sub, T_delta, epochs=100)
third_run.train(X_sub, T_delta, epochs=100)

# plot metrics for all three algorithms on the same plot

plot_metrics_comparison_multiple_run(first_run=first_run, second_run=second_run, third_run=third_run, saved_fig_name="removed_25_A_25_B.png")

In [ ]:
# remove 50% of the data points from class A

idx_A = np.where(T_og == 1)[0]
idx_B = np.where(T_og == 0)[0]

# Rimuovo il 50% di A
remove_A = np.random.choice(
    idx_A,
    size=int(0.50 * len(idx_A)),
    replace=False
)

keep_A = np.setdiff1d(idx_A, remove_A)

# Tengo tutta B
keep_idx = np.concatenate([keep_A, idx_B])

X_sub = X_og[:, keep_idx]
T_sub = T_og[keep_idx]

T_delta = np.where(T_sub == 0, -1, T_sub)

plot_data(X_sub, T_sub, title="Second Non-Linearly Separable Dataset")
first_run.train(X_sub, T_delta, epochs=100)
second_run.train(X_sub, T_delta, epochs=100)
third_run.train(X_sub, T_delta, epochs=100)

# plot metrics for all three algorithms on the same plot

plot_metrics_comparison_multiple_run(first_run=first_run, second_run=second_run, third_run=third_run, saved_fig_name="removed_50_A.png")

In [ ]:
# remove 50% of the data points from class B

idx_A = np.where(T_og == 1)[0]
idx_B = np.where(T_og == 0)[0]

# Rimuovo il 50% di B
remove_B = np.random.choice(
    idx_B,
    size=int(0.50 * len(idx_B)),
    replace=False
)

keep_B = np.setdiff1d(idx_B, remove_B)

# Tengo tutta A
keep_idx = np.concatenate([idx_A, keep_B])

X_sub = X_og[:, keep_idx]
T_sub = T_og[keep_idx]

T_delta = np.where(T_sub == 0, -1, T_sub)

plot_data(X_sub, T_sub, title="Second Non-Linearly Separable Dataset")
first_run.train(X_sub, T_delta, epochs=100)
second_run.train(X_sub, T_delta, epochs=100)
third_run.train(X_sub, T_delta, epochs=100)

# plot metrics for all three algorithms on the same plot

plot_metrics_comparison_multiple_run(first_run=first_run, second_run=second_run, third_run=third_run, saved_fig_name="removed_50_B.png")

In [ ]:
# remove 20% of the data points from class A where A(0, :) < 0 and 80% of the data points from class A where A(0, :) > 0

idx_A = np.where(T_og == 1)[0]
idx_B = np.where(T_og == 0)[0]

# Divido A in base a x1
idx_A_left = idx_A[X_og[0, idx_A] < 0]
idx_A_right = idx_A[X_og[0, idx_A] > 0]

# Rimuovo il 20% da A con x1 < 0
remove_left = np.random.choice(
    idx_A_left,
    size=int(0.20 * len(idx_A_left)),
    replace=False
)

# Rimuovo l'80% da A con x1 > 0
remove_right = np.random.choice(
    idx_A_right,
    size=int(0.8 * len(idx_A_right)),
    replace=False
)

remove_A = np.concatenate([remove_left, remove_right])

# Tengo tutti i punti che non ho rimosso
keep_A = np.setdiff1d(idx_A, remove_A)

# B rimane completamente
keep_idx = np.concatenate([keep_A, idx_B])

X_sub = X_og[:, keep_idx]
T_sub = T_og[keep_idx]

#shuffle the data points indexes

shuffle_idx = np.random.permutation(X_sub.shape[1])
X_sub = X_sub[:, shuffle_idx]
T_sub = T_sub[shuffle_idx]



T_delta = np.where(T_sub == 0, -1, T_sub)

plot_data(X_sub, T_sub, title="Second Non-Linearly Separable Dataset")



first_run.train(X_sub, T_delta, epochs=500)
second_run.train(X_sub, T_delta, epochs=500)
third_run.train(X_sub, T_delta, epochs=500)

# plot metrics for all three algorithms on the same plot

plot_metrics_comparison_multiple_run(first_run=first_run, second_run=second_run, third_run=third_run, plot_classes_accuracy=True, saved_fig_name="removed_20_A_neg_80_A_pos.png")